# Ground Truth Measurement Generation
This python notebook is used to show how the ground truth measurements were generated to validate the parallel and pytorch implimentations of the GSTATSIM interpolation functions

## Initialize data and simulation grid

In [1]:
import sys
sys.path.append("../")

In [24]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import QuantileTransformer
import skgstat as skg
from skgstat import models
import gstatsim as gs
from sklearn.cluster import KMeans

In [3]:
# load data
df_bed = pd.read_csv('../demos/data/greenland_test_data.csv')
# remove erroneously high values due to bad bed picks
df_bed = df_bed[df_bed["Bed"] <= 700]  

In [4]:
# grid data to 100 m resolution and remove coordinates with NaNs
res = 1000
df_grid, grid_matrix, rows, cols = gs.Gridding.grid_data(df_bed, 'X', 'Y', 'Bed', res)
df_grid = df_grid[df_grid["Z"].isnull() == False]
df_grid = df_grid.rename(columns = {"Z": "Bed"})

# normal score transformation
data = df_grid['Bed'].values.reshape(-1,1)
nst_trans = QuantileTransformer(n_quantiles=500, output_distribution="normal").fit(data)
df_grid['Nbed'] = nst_trans.transform(data) 

# compute experimental (isotropic) variogram
coords = df_grid[['X','Y']].values
values = df_grid['Nbed']

maxlag = 50000             # maximum range distance
n_lags = 70                # num of bins

V1 = skg.Variogram(coords, values, bin_func='even', n_lags=n_lags, 
                   maxlag=maxlag, normalize=False)

# use exponential variogram model
V1.model = 'exponential'
V1.parameters

[31852.813365632115, 0.7027482242025527, 0]

In [5]:
# define coordinate grid
xmin = np.min(df_grid['X']); xmax = np.max(df_grid['X'])     # min and max x values
ymin = np.min(df_grid['Y']); ymax = np.max(df_grid['Y'])     # min and max y values

# initialize grid
Pred_grid_xy = gs.Gridding.prediction_grid(xmin, xmax, ymin, ymax, res)

In [6]:
# set variogram parameters
azimuth = 0
nugget = V1.parameters[2]

# the major and minor ranges are the same in this example because it is isotropic
major_range = V1.parameters[0]
minor_range = V1.parameters[0]
sill = V1.parameters[1]
vtype = 'Exponential'

# save variogram parameters as a list
vario = [azimuth, nugget, major_range, minor_range, sill, vtype]


k = 50         # number of neighboring data points used to estimate a given point
rad = 50000     # 50 km search radius

## Simple and Ordinary Kriging
Since Simple Kriging and Ordinary Kriging are both deterministic processes, we only need to generate one topographic model to serve as the ground truth

In [7]:
est_SK, var_SK = gs.Interpolation.skrige(Pred_grid_xy, df_grid, 'X', 'Y', 'Nbed', k, vario, rad)

100%|██████████| 22500/22500 [00:56<00:00, 396.16it/s]


In [14]:
GT_skrig = np.column_stack((est_SK, var_SK))
np.savetxt("GT/skrig.csv", GT_skrig, delimiter=",")

In [15]:
est_OK, var_OK = gs.Interpolation.okrige(Pred_grid_xy, df_grid, 'X', 'Y', 'Nbed', k, vario, rad)

100%|██████████| 22500/22500 [00:56<00:00, 395.36it/s]


In [16]:
GT_okrig = np.column_stack((est_OK, var_OK))
np.savetxt("GT/okrig.csv", GT_okrig, delimiter=",")

## SGS with Simple and Ordinary Kriging
Since SGS is a stochastic process, we will generate 10 realizations to use as a source of comparison with the parallelized implimentation

In [15]:
# initalize array to store multiple realizations
SGS_SK_batch = np.zeros((len(Pred_grid_xy),10))

In [16]:
for i in range(10):
    print(f'Generating realization #{i+1}:')
    SGS_SK_batch[:,i] = gs.Interpolation.skrige_sgs(Pred_grid_xy, df_grid, 'X', 'Y', 'Nbed', k, vario, rad)

Generating realization #1:


100%|██████████| 22500/22500 [01:17<00:00, 290.74it/s]


Generating realization #2:


100%|██████████| 22500/22500 [01:18<00:00, 288.01it/s]


Generating realization #3:


...

100%|██████████| 22500/22500 [01:17<00:00, 289.23it/s]


Generating realization #9:


100%|██████████| 22500/22500 [03:47<00:00, 98.90it/s]  


Generating realization #10:


100%|██████████| 22500/22500 [01:22<00:00, 273.82it/s]


In [17]:
np.savetxt("GT/sgs_skrig.csv", SGS_SK_batch, delimiter=",")

In [18]:
# initalize array to store multiple realizations
SGS_OK_batch = np.zeros((len(Pred_grid_xy),10))

In [19]:
for i in range(10):
    print(f'Generating realization #{i+1}:')
    SGS_OK_batch[:,i] = gs.Interpolation.okrige_sgs(Pred_grid_xy, df_grid, 'X', 'Y', 'Nbed', k, vario, rad)

Generating realization #1:


100%|██████████| 22500/22500 [01:21<00:00, 275.43it/s]


Generating realization #2:


100%|██████████| 22500/22500 [01:23<00:00, 269.23it/s]


Generating realization #3:


...

100%|██████████| 22500/22500 [01:22<00:00, 274.23it/s]


Generating realization #9:


100%|██████████| 22500/22500 [01:21<00:00, 275.08it/s]


Generating realization #10:


100%|██████████| 22500/22500 [01:22<00:00, 272.29it/s]


In [20]:
np.savetxt("GT/sgs_okrig.csv", SGS_OK_batch, delimiter=",")

## Cluster SGS

In [29]:
# K means clustering
n_clusters = 3
kmeans = KMeans(n_clusters = n_clusters, random_state = 0, n_init = 10).fit(df_grid[['X','Y','Nbed']])
df_grid['K'] = kmeans.labels_  # make column in dataframe with cluster name

# experimental variogram parameters
maxlag = 50000
n_lags = 70 #num of bins

# cluster 0 variogram
df0 = df_grid[df_grid['K'] == 0]
coords0 = df0[['X','Y']].values
values0 = df0['Nbed']
V0 = skg.Variogram(coords0, values0, bin_func = "even", n_lags = n_lags, 
                   maxlag = maxlag, normalize=False)


# cluster 1 variogram
df1 = df_grid[df_grid['K'] == 1]
coords1 = df1[['X','Y']].values
values1 = df1['Nbed']
V1 = skg.Variogram(coords1, values1, bin_func = "even", n_lags = n_lags, 
                   maxlag = maxlag, normalize=False)


# cluster 2 variogram
df2 = df_grid[df_grid['K'] == 2]
coords2 = df2[['X','Y']].values
values2 = df2['Nbed']
V2 = skg.Variogram(coords2, values2, bin_func = "even", n_lag = n_lags, 
                   maxlag = maxlag, normalize=False) 

range0 = V0.parameters[0]; sill0 = V0.parameters[1]
range1 = V1.parameters[0]; sill1 = V1.parameters[1]
range2 = V2.parameters[0]; sill2 = V2.parameters[1]

# make a list with variogram parameters
azimuth = 0

# nugget effect
nug = 0 

# variogram model
vtype = 'Exponential'

# define variograms for each cluster
# Azimuth, nugget, major range, minor range, sill
gam0 = [azimuth, nug, range0, range0, sill0, vtype]
gam1 = [azimuth, nug, range1, range1, sill1, vtype]
gam2 = [azimuth, nug, range2, range2, sill2, vtype]

# store variogram parameters
df_gamma = pd.DataFrame({'Variogram': [gam0, gam1, gam2]})


In [30]:
# initalize array to store multiple realizations
SGS_cluster_batch = np.zeros((len(Pred_grid_xy),10))

In [31]:
for i in range(10):
    print(f'Generating realization #{i+1}:')
    SGS_cluster_batch[:,i] = gs.Interpolation.cluster_sgs(Pred_grid_xy, df_grid, 'X', 'Y', 'Nbed', 'K', k, df_gamma, rad)

Generating realization #1:


100%|██████████| 22500/22500 [01:29<00:00, 251.18it/s]


Generating realization #2:


100%|██████████| 22500/22500 [01:30<00:00, 249.14it/s]


Generating realization #3:


...

100%|██████████| 22500/22500 [01:30<00:00, 248.90it/s]


Generating realization #9:


100%|██████████| 22500/22500 [01:29<00:00, 250.15it/s]


Generating realization #10:


100%|██████████| 22500/22500 [01:28<00:00, 254.92it/s]


In [32]:
np.savetxt("GT/sgs_cluster.csv", SGS_cluster_batch, delimiter=",")